# Mirror circuits

In [ ]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

In [28]:
import time
import numpy as np
from qiskit import *
from qiskit.quantum_info import hellinger_fidelity

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit_device_benchmarking.mirror_test.mirror_test import submit_mirror_test, analyze_mirror_result
from qiskit_device_benchmarking.mirror_test.mirror_circuits import trotter_circuit
import matplotlib.pyplot as plt

In [29]:
import json
from datetime import datetime

def default(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat(),
        }
    raise TypeError(f"Cannot serialize {type(obj)}")

def object_hook(obj):
    if obj.get("__type__") == "datetime":
        return datetime.fromisoformat(obj["value"])
    return obj

data = {
    "name": "test",
    "created": datetime.now(),
}

In [30]:
def polarization_fidelity(success_prob, num_qubits):
    """From metriq-gym

    https://github.com/unitaryfoundation/metriq-gym/metriq_gym/benchmarks/mirror_circuits.py
    """
    baseline = 1.0 / (2**num_qubits)
    polarization = (
            (success_prob - baseline) / (1.0 - baseline)
            if ((1.0 - baseline) > 0) and ((success_prob - baseline) > 0)
            else 0.0
        )
    return polarization

In [31]:
service = QiskitRuntimeService()

backend = service.backend("ibm_chicago", use_fractional_gates=True)

In [33]:
qubit_list = list(range(10,85,5))
sampler = Sampler(backend)
shots = 10_000

/tmp/ipykernel_45795/1643073582.py:3: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(backend)


In [34]:
jobs = []
for ii in qubit_list:
    for jj in qubit_list:
        num_qubits = ii
        num_steps = jj
        cmap = [(kk,kk+1) for kk in range(0,num_qubits-1)]

        qc = trotter_circuit(np.pi/8,layer_couplings=[cmap],num_steps=num_steps,gate_2q='cz')
        mirror_qc = qc.compose(qc.inverse())
        mirror_qc.measure_all()
        trans_mirror_qc = transpile(mirror_qc, backend, optimization_level=3)
        job = sampler.run([trans_mirror_qc], shots=shots)
        jobs.append(job)

In [40]:
results= {}
job_times = []
counter = 0
st = time.perf_counter()
for ii in qubit_list:
    for jj in qubit_list:
        num_qubits = ii
        num_steps = jj
        job = jobs[counter]
        job_times.append(job.usage())
        counter += 1
        ans = {'0'*num_qubits: 1.0}
        counts = job.result()[0].join_data().get_counts()
        fid = hellinger_fidelity(counts, ans)
        final_err = np.sqrt(fid * (1.0 - fid) / shots) 
        polar = polarization_fidelity(fid, num_qubits)
        polar_err = float(final_err / (1.0 - 1/2.0**num_qubits) if polar > 0 else 0.0)
        results[f"({ii},{jj})"] = [fid, polar, polar_err, polar > 1/np.e]

ft = time.perf_counter()
classical_time = ft - st

In [20]:
res = {'name': 'mirror', 'backend': backend.name, 'datetime': str(jobs[0].creation_date),
           'backend_properties': json.dumps(backend.properties().to_dict(), default=default),
           'shots': shots, 'quantum_runtime': sum(mm.usage() for mm in jobs),
            'jobs_time': job_times, 'result': results}

In [22]:
with open("mirror_results.json", "w") as f:
        json.dump(res, f)

In [23]:
num_qubits = 30
num_steps = 50
cmap = [(kk,kk+1) for kk in range(0,num_qubits-1)]

qc = trotter_circuit(np.pi/8,layer_couplings=[cmap],num_steps=num_steps,gate_2q='cz')
mirror_qc = qc.compose(qc.inverse())
mirror_qc.measure_all()
trans_mirror_qc = transpile(mirror_qc, backend, optimization_level=3)

In [24]:
trans_mirror_qc.count_ops()

OrderedDict([('rz', 4740),
             ('rx', 3000),
             ('cz', 2400),
             ('rzz', 500),
             ('barrier', 201),
             ('measure', 30)])